# ⚡ Zalo AI Challenge 2026: DataMate — SFT Dataset Expansion (Kaggle 2 × T4 GPUs)
## Sinh thêm Dataset SFT chuẩn (+1đ) bằng Qwen 7B + LoRA Adapter trên Kaggle 2 × T4

> **Cấu hình & Tối ưu hóa đặc biệt cho Kaggle 2 × T4 (30GB VRAM tổng cộng):**
> - **Tensor Parallel (`tp=2`):** vLLM phân bổ mô hình Qwen 7B đều sang cả 2 GPU (mỗi GPU chỉ chiếm ~7GB VRAM / 15GB), dư 8GB VRAM cho KV cache và LoRA.
> - **Precision (`float16`):** Tối ưu 100% cho Tensor Cores kiến trúc Turing của GPU T4 (tránh lỗi và overhead của `bfloat16` trên T4).
> - **LoRA Adapter Injection:** Sử dụng chính Adapter LoRA đã huấn luyện (`final_adapter`) để sinh code giải các câu hỏi trong tập train, sau đó đưa qua bộ lọc 4 Cổng khắt khe (+1.0 điểm chuẩn BTC, AST đọc file thật, exit 0).
> - **Kaggle Auto-Discovery:** Tự động nhận diện đường dẫn dữ liệu `data/train` và adapter LoRA dù bạn để ở `/kaggle/input/...` hay `/kaggle/working`.

### Bước 1: Kiểm tra cấu hình GPU Kaggle (2 × Tesla T4)
Xác nhận môi trường Kaggle đang chạy 2 GPU T4 (Kaggle Settings -> Accelerator -> GPU T4 x2).

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), '❌ CUDA không khả dụng! Vui lòng bật GPU T4 x2 trong Settings bên phải của Kaggle.'
num_gpus = torch.cuda.device_count()
print(f'\n🔥 Số lượng GPU phát hiện: {num_gpus}')

for i in range(num_gpus):
    gpu_name = torch.cuda.get_device_name(i)
    total_mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f'   - GPU {i}: {gpu_name} ({total_mem:.2f} GB VRAM)')

if num_gpus < 2:
    print('\n⚠️ Chú ý: Hiện tại chỉ phát hiện 1 GPU. Để tận dụng tối đa tốc độ, hãy vào Settings -> Accelerator và chọn "GPU T4 x2".')
else:
    print('\n✅ Đã sẵn sàng chạy Tensor Parallel (TP=2) trên 2 × Tesla T4 GPUs!')

### Bước 2: Cài đặt Dependencies cho vLLM & Sandbox
Cài đặt thư viện xử lý bảng tính (`calamine`, `openpyxl`), `vllm`, `peft` và `transformers`.

In [ ]:
# Cài đặt các thư viện cần thiết cho DataMate sandbox
# Gỡ bỏ torchao xung đột phiên bản với PEFT
!pip uninstall -y torchao -q
!pip install -q "pandas>=2.2.0" openpyxl python-calamine xlrd duckdb pyyaml tqdm pydantic

import torch, transformers, peft
print(f'✅ Môi trường đã sẵn sàng! PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | PEFT: {peft.__version__}')


### Bước 3: Thiết lập Workspace & Tự động dò tìm Data/Adapter
Kaggle tổ chức file đầu vào tại `/kaggle/input` (chỉ đọc) và thư mục làm việc tại `/kaggle/working` (có quyền ghi).
Cell này tự động quét và thiết lập:

In [ ]:
import os, sys, shutil
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')

# 1. Dò tìm mã nguồn dự án (PROJECT_ROOT) quét đệ quy toàn bộ /kaggle/input
candidate_roots = [
    KAGGLE_WORKING / 'datamate',
    KAGGLE_WORKING / 'ZALO-AI-CHALLENGE-2026',
    KAGGLE_WORKING,
    Path('.').resolve(),
]
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.glob('**'):
        if p.is_dir() and (p / 'scripts' / 'generate_sft_vllm.py').exists() and (p / 'src').exists():
            candidate_roots.insert(0, p)
            break

PROJECT_ROOT = None
for cand in candidate_roots:
    if (cand / 'scripts' / 'generate_sft_vllm.py').exists():
        PROJECT_ROOT = cand
        break

# Nếu mã nguồn đang nằm trong /kaggle/input (chỉ đọc), đồng bộ sang /kaggle/working/datamate
if PROJECT_ROOT and str(PROJECT_ROOT).startswith('/kaggle/input'):
    print(f'📦 Đang đồng bộ mã nguồn từ {PROJECT_ROOT} sang /kaggle/working/datamate...')
    work_dir = KAGGLE_WORKING / 'datamate'
    work_dir.mkdir(parents=True, exist_ok=True)
    !rsync -a --exclude='.git' --exclude='tmp' --exclude='wandb' "{PROJECT_ROOT}/" "{work_dir}/"
    PROJECT_ROOT = work_dir

assert PROJECT_ROOT is not None and (PROJECT_ROOT / 'scripts' / 'generate_sft_vllm.py').exists(), \
    '❌ Không tìm thấy scripts/generate_sft_vllm.py! Hãy kiểm tra dataset đầu vào trên Kaggle.'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print(f'✅ Thư mục làm việc: {PROJECT_ROOT}')

# 2. Dò tìm thư mục dữ liệu data/train
data_train_candidates = [
    PROJECT_ROOT / 'data' / 'train',
    KAGGLE_WORKING / 'data' / 'train',
]
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.glob('**'):
        if p.is_dir() and (p / 'questions.jsonl').exists() and (p / 'answers.jsonl').exists():
            data_train_candidates.insert(0, p)
            break

DATA_TRAIN_DIR = None
for cand in data_train_candidates:
    if (cand / 'questions.jsonl').exists() and (cand / 'answers.jsonl').exists():
        DATA_TRAIN_DIR = cand
        break

assert DATA_TRAIN_DIR is not None, '❌ Không tìm thấy thư mục data/train (cần có questions.jsonl và answers.jsonl)!'
print(f'✅ Thư mục data/train: {DATA_TRAIN_DIR}')

# Nếu data/train không nằm trong PROJECT_ROOT/data/train, tạo symlink để script đọc trực tiếp
local_data_train = PROJECT_ROOT / 'data' / 'train'
if not (local_data_train / 'questions.jsonl').exists():
    local_data_train.parent.mkdir(parents=True, exist_ok=True)
    if local_data_train.exists():
        shutil.rmtree(local_data_train)
    os.symlink(str(DATA_TRAIN_DIR), str(local_data_train))
    print(f'🔗 Đã liên kết data/train -> {DATA_TRAIN_DIR}')

# 3. Dò tìm LoRA Adapter (final_adapter)
adapter_candidates = [
    PROJECT_ROOT / 'result/sft_adapter_qwen7b/final_adapter',
    PROJECT_ROOT / 'result/sft_adapter_qwen7b',
    KAGGLE_WORKING / 'sft_adapter_qwen7b/final_adapter',
    KAGGLE_WORKING / 'final_adapter',
]
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.glob('**'):
        if p.is_dir() and (p / 'adapter_config.json').exists():
            adapter_candidates.insert(0, p)
            break

CHOSEN_ADAPTER = None
for cand in adapter_candidates:
    if (cand / 'adapter_config.json').exists():
        CHOSEN_ADAPTER = cand
        break

if CHOSEN_ADAPTER:
    print(f'🎯 Đã tìm thấy LoRA Adapter: {CHOSEN_ADAPTER}')
else:
    print('⚠️ Không tìm thấy LoRA Adapter; sẽ sinh code bằng base model Qwen 7B.')

# 4. Tự động kiểm tra và vá nóng scripts/generate_sft_vllm.py (Cân bằng bộ nhớ 2 x T4, tránh lỗi CUDA OOM)
script_target = PROJECT_ROOT / 'scripts' / 'generate_sft_vllm.py'
if script_target.exists():
    code_text = script_target.read_text(encoding='utf-8')
    needs_patch = ('max_memory=max_mem_dict' not in code_text) or ('for prompt_text in prompts:' not in code_text)
    if needs_patch:
        print('⚡ Đang áp dụng bản vá cân bằng bộ nhớ VRAM cho Kaggle 2 × T4 vào generate_sft_vllm.py...')
        old_part1 = '''base_model_obj = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch_dtype,
            device_map="auto",
            trust_remote_code=True,
        )'''
        new_part1 = '''n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
        max_mem_dict = {0: "7.0GiB", 1: "14.0GiB"} if n_gpus > 1 else None
        if max_mem_dict:
            print(f"⚖️ Phân bổ bộ nhớ đa GPU: {max_mem_dict}")
        base_model_obj = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch_dtype,
            device_map="auto",
            max_memory=max_mem_dict,
            attn_implementation="sdpa",
            trust_remote_code=True,
        )'''
        if old_part1 in code_text:
            code_text = code_text.replace(old_part1, new_part1)

        old_part2 = '''        else:
            import torch
            outputs = []
            tf_bs = min(len(prompts), 8)
            dev = next(model_obj.parameters()).device
            for b_idx in range(0, len(prompts), tf_bs):
                sub_prompts = prompts[b_idx:b_idx + tf_bs]
                inputs = tokenizer(sub_prompts, return_tensors="pt", padding=True, truncation=True,
                                   max_length=max_allowed_prompt_len).to(dev)
                with torch.no_grad():
                    gen_ids = model_obj.generate(
                        **inputs,
                        max_new_tokens=args.max_tokens,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                input_lens = [len(inp) for inp in inputs.input_ids]
                for sub_i, out_seq in enumerate(gen_ids):
                    gen_tokens = out_seq[input_lens[sub_i]:]
                    text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
                    outputs.append(DummyOutput(text))'''

        new_part2 = '''        else:
            import torch
            outputs = []
            dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
            for prompt_text in prompts:
                inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True,
                                   max_length=max_allowed_prompt_len).to(dev)
                with torch.inference_mode():
                    gen_ids = model_obj.generate(
                        **inputs,
                        max_new_tokens=args.max_tokens,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                gen_tokens = gen_ids[0][inputs.input_ids.shape[1]:]
                text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
                outputs.append(DummyOutput(text))
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()'''
        if old_part2 in code_text:
            code_text = code_text.replace(old_part2, new_part2)

        script_target.write_text(code_text, encoding='utf-8')
        print('✅ Đã vá xong generate_sft_vllm.py! Sẵn sàng chạy an toàn 100% trên 2 × T4.')


### Bước 4: Chạy Sinh Dataset SFT với vLLM (2 × T4 GPUs, TP=2, float16)
- Sử dụng `--tensor-parallel-size 2` để phân bổ Qwen 7B đều sang 2 GPU T4.
- Sử dụng `--dtype float16` tối ưu cho Turing architecture của T4.
- Sử dụng `--lora-adapter` đã huấn luyện để tăng tỷ lệ code đúng nghiệp vụ.
- Kết quả được lưu trực tiếp vào `/kaggle/working/result/sft_dataset_qwen7b_lora_t4x2/`.

In [ ]:
import subprocess, sys
from pathlib import Path

output_dir = KAGGLE_WORKING / 'result' / 'sft_dataset_qwen7b_lora_t4x2'
output_dir.mkdir(parents=True, exist_ok=True)

adapter_args = ['--lora-adapter', str(CHOSEN_ADAPTER)] if CHOSEN_ADAPTER else []

# Chạy generator với backend auto (tự động dùng Transformers + PEFT device_map='auto' phân bổ đều 2 GPU T4)
command = [
    sys.executable, '-u', 'scripts/generate_sft_vllm.py',
    '--backend', 'auto',
    '--model-id', 'Qwen/Qwen2.5-Coder-7B-Instruct',
    '--output-dir', str(output_dir),
    '--dtype', 'float16',                  # float16 tối ưu cho Tensor Cores T4
    '--batch-size', '16',
    '--max-model-len', '16384',
    '--max-tokens', '2048',
    '--sandbox-workers', '2',
    '--sandbox-timeout', '35',
] + adapter_args

print('🚀 Khởi chạy SFT Generator...')
print('Lệnh thực thi:', ' '.join(command), '\n')

run_env = os.environ.copy()
run_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

with subprocess.Popen(command, cwd=PROJECT_ROOT, env=run_env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace') as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise

if return_code:
    raise RuntimeError(f'Quá trình sinh dataset thất bại (exit {return_code}). Xem log ở trên.')

print('\n🎉 Quá trình sinh dataset trên Kaggle 2 × T4 đã hoàn tất!')


### Bước 5: Kiểm tra Thống kê & Xuất Dataset Đạt Chuẩn (+1đ)
Hiển thị số lượng mẫu đạt chuẩn đã thu thập được và hướng dẫn tải về.

In [ ]:
import json
from pathlib import Path

sft_file = output_dir / 'verified_sft_dataset.jsonl'

if sft_file.exists():
    with open(sft_file, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()]
    print(f'\n🎉 TỔNG CỘNG ĐÃ SINH & LỌC ĐƯỢC: {len(records)} MẪU SFT ĐẠT CHUẨN (+1Đ)!')
    if records:
        print(f'\n--- Xem trước mẫu mới nhất (ID: {records[-1].get("id")}) ---')
        msgs = records[-1].get('messages', [])
        if len(msgs) >= 3:
            print('User Prompt trích đoạn:', msgs[1]['content'][:200], '...')
            print('\nCode Assistant sinh ra:\n', msgs[2]['content'][:300], '...')
        print(f'\n📂 File dataset lưu tại: {sft_file.resolve()}')
        print('💡 Bạn có thể tải file này trực tiếp từ tab "Output" bên phải giao diện Kaggle!')
else:
    print(f'⚠️ Chưa tìm thấy file {sft_file}')